
# SD v1.5 + ControlNet + LoRA 기반 한국 화풍 스케치 생성 학습

이 노트북은 현재 1단계만 구현합니다.

- 입력: 텍스트 설명 + 스케치
- 출력: 한국 전통 화풍 이미지
- 학습 1: 원본 이미지와 설명으로 SD v1.5 스타일 LoRA 학습
- 학습 2: 원본 이미지, 스케치, 설명으로 SD v1.5 ControlNet 학습
- 추론: SD v1.5 + 학습된 ControlNet + 학습된 LoRA 결합

VLM 기반 reference 선택, IP-Adapter, evaluator, weighted final selection은 이번 노트북에서 제외합니다.

기본 데이터 구조는 다음과 같습니다.

```text
MyDrive/data/
├── sketch/
│   ├── 02575.png
│   └── ...
└── image_txt/
    ├── 02575.png
    ├── 02575.txt
    └── ...
```



## 실행 전 준비

1. Colab GPU 런타임을 선택합니다.
2. SD v1.5 모델 접근이 필요한 경우 Hugging Face에서 모델 약관을 승인합니다.
3. 아래 설정 셀에서 `DATA_ROOT`를 실제 Drive 경로에 맞게 수정합니다.
4. 기본값은 Colab 16GB급 GPU를 고려해 512 해상도와 Canny ControlNet을 사용합니다.


In [ ]:

import os
import shutil
import subprocess
import sys
from pathlib import Path

DIFFUSERS_REPO = Path("/content/diffusers")
DIFFUSERS_TAG = "v0.39.0"

if not DIFFUSERS_REPO.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--quiet",
            "--depth",
            "1",
            "--branch",
            DIFFUSERS_TAG,
            "https://github.com/huggingface/diffusers.git",
            str(DIFFUSERS_REPO),
        ],
        check=True,
    )

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(DIFFUSERS_REPO)], check=True)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(DIFFUSERS_REPO / "examples/controlnet/requirements.txt"),
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(DIFFUSERS_REPO / "examples/text_to_image/requirements.txt"),
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "bitsandbytes>=0.46.0",
        "safetensors",
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "xformers", "--no-deps"],
    check=False,
)

print("설치 완료")


In [ ]:

import importlib.util
import json
import math
import os
import random
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

import torch
from PIL import Image, ImageOps
from IPython.display import display

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError("GPU 런타임을 선택해 주세요.")


In [ ]:

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    print("Colab이 아니므로 Drive 마운트를 건너뜁니다.")

from huggingface_hub import notebook_login

notebook_login()


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/BUHT")

DATA_ROOT = PROJECT_ROOT / "data"
SKETCH_DIR = DATA_ROOT / "sketch"
TARGET_DIR = DATA_ROOT / "image_txt"

WORK_ROOT = Path("/content/korean_sketch_project")
DATASET_DIR = WORK_ROOT / "dataset"

OUTPUT_ROOT = Path("/content/drive/MyDrive/korean_sketch_model")
LORA_OUTPUT_DIR = OUTPUT_ROOT / "lora"
CONTROLNET_OUTPUT_DIR = OUTPUT_ROOT / "controlnet"
GENERATED_DIR = OUTPUT_ROOT / "generated"

BASE_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"
CONTROLNET_INIT = "lllyasviel/sd-controlnet-canny"

RESOLUTION = 512
SEED = 42
VAL_RATIO = 0.05
STYLE_PREFIX = "Korean traditional painting style, "
STRICT_SIZE_MATCH = True
REBUILD_DATASET = True

LORA_EPOCHS = 5
CONTROLNET_EPOCHS = 10
LORA_RANK = 16
LORA_LR = 1e-4
CONTROLNET_LR = 1e-5
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
CHECKPOINTING_STEPS = 500
CHECKPOINTS_TOTAL_LIMIT = 3

TRAIN_LORA = True
TRAIN_CONTROLNET = True
RESUME_LORA = False
RESUME_CONTROLNET = False
ENABLE_TRAIN_VALIDATION = False

MIXED_PRECISION = "fp16"
USE_8BIT_ADAM = True

try:
    import xformers
    USE_XFORMERS = True
except Exception as xformers_error:
    print("xformers 비활성화:", xformers_error)
    USE_XFORMERS = False

WORK_ROOT.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LORA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONTROLNET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

assert PROJECT_ROOT.exists(), f"BUHT 폴더가 없습니다: {PROJECT_ROOT}"
assert SKETCH_DIR.exists(), f"스케치 폴더가 없습니다: {SKETCH_DIR}"
assert TARGET_DIR.exists(), f"이미지 및 텍스트 폴더가 없습니다: {TARGET_DIR}"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SKETCH_DIR:", SKETCH_DIR)
print("TARGET_DIR:", TARGET_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("USE_XFORMERS:", USE_XFORMERS)

In [ ]:

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}

def index_images(folder):
    result = {}
    for path in folder.iterdir():
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            result[path.stem] = path
    return result

def index_texts(folder):
    return {path.stem: path for path in folder.glob("*.txt") if path.is_file()}

def read_caption(path):
    for encoding in ("utf-8-sig", "utf-8", "cp949"):
        try:
            return path.read_text(encoding=encoding)
        except UnicodeDecodeError:
            pass
    return path.read_text(encoding="utf-8", errors="replace")

def clean_caption(text):
    compact = " ".join(text.split())
    return f"{STYLE_PREFIX}{compact}".strip()

def image_to_rgb(image, fill):
    image = ImageOps.exif_transpose(image)
    if image.mode in ("RGBA", "LA") or "transparency" in image.info:
        background = Image.new("RGBA", image.size, fill + (255,))
        background.alpha_composite(image.convert("RGBA"))
        return background.convert("RGB")
    return image.convert("RGB")

def save_square_image(source, destination, size, fill):
    with Image.open(source) as image:
        image = image_to_rgb(image, fill)
        image = ImageOps.pad(
            image,
            (size, size),
            method=Image.Resampling.LANCZOS,
            color=fill,
            centering=(0.5, 0.5),
        )
        image.save(destination, format="PNG", optimize=True)

target_index = index_images(TARGET_DIR)
sketch_index = index_images(SKETCH_DIR)
text_index = index_texts(TARGET_DIR)

paired_ids = sorted(set(target_index) & set(sketch_index) & set(text_index))

missing_target = sorted((set(sketch_index) & set(text_index)) - set(target_index))
missing_sketch = sorted((set(target_index) & set(text_index)) - set(sketch_index))
missing_text = sorted((set(target_index) & set(sketch_index)) - set(text_index))

if not paired_ids:
    raise RuntimeError("같은 파일명을 가진 원본 이미지, 스케치, txt 조합을 찾지 못했습니다.")

rng = random.Random(SEED)
shuffled_ids = paired_ids.copy()
rng.shuffle(shuffled_ids)

if len(shuffled_ids) == 1:
    train_ids = shuffled_ids
    val_ids = shuffled_ids
else:
    val_count = max(1, int(round(len(shuffled_ids) * VAL_RATIO)))
    val_count = min(val_count, len(shuffled_ids) - 1)
    val_ids = shuffled_ids[:val_count]
    train_ids = shuffled_ids[val_count:]

print("전체 페어:", len(paired_ids))
print("학습:", len(train_ids))
print("검증:", len(val_ids))
print("원본 누락 예시:", missing_target[:10])
print("스케치 누락 예시:", missing_sketch[:10])
print("텍스트 누락 예시:", missing_text[:10])


In [ ]:
import os
import json
import shutil
from tqdm.auto import tqdm

def create_link(source, destination):
    source = Path(source).resolve()
    destination = Path(destination)

    if destination.exists() or destination.is_symlink():
        destination.unlink()

    os.symlink(source, destination)

def prepare_split(split_name, stems):
    split_dir = DATASET_DIR / split_name
    target_out = split_dir / "target"
    sketch_out = split_dir / "sketch"

    if REBUILD_DATASET and split_dir.exists():
        shutil.rmtree(split_dir)

    target_out.mkdir(parents=True, exist_ok=True)
    sketch_out.mkdir(parents=True, exist_ok=True)

    metadata_path = split_dir / "metadata.jsonl"
    records = []

    for stem in tqdm(stems, desc=f"{split_name} 메타데이터 생성"):
        target_path = target_index[stem]
        sketch_path = sketch_index[stem]

        target_name = f"{stem}{target_path.suffix.lower()}"
        sketch_name = f"{stem}{sketch_path.suffix.lower()}"

        create_link(target_path, target_out / target_name)
        create_link(sketch_path, sketch_out / sketch_name)

        caption = clean_caption(read_caption(text_index[stem]))

        records.append(
            {
                "image_file_name": f"target/{target_name}",
                "conditioning_image_file_name": f"sketch/{sketch_name}",
                "text": caption,
                "id": stem,
            }
        )

    with metadata_path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")

    return records

train_records = prepare_split("train", train_ids)
validation_records = prepare_split("validation", val_ids)

config_snapshot = {
    "data_root": str(DATA_ROOT),
    "base_model": BASE_MODEL,
    "controlnet_init": CONTROLNET_INIT,
    "resolution": RESOLUTION,
    "seed": SEED,
    "style_prefix": STYLE_PREFIX,
    "train_count": len(train_records),
    "validation_count": len(validation_records),
}

with (OUTPUT_ROOT / "training_config.json").open("w", encoding="utf-8") as file:
    json.dump(config_snapshot, file, ensure_ascii=False, indent=2)

print("데이터셋 연결 완료:", DATASET_DIR)
print("학습 데이터:", len(train_records))
print("검증 데이터:", len(validation_records))

In [ ]:

from datasets import load_dataset

dataset = load_dataset("imagefolder", data_dir=str(DATASET_DIR))
print(dataset)
print(dataset["train"].features)

required_columns = {"image", "conditioning_image", "text"}
missing_columns = required_columns - set(dataset["train"].column_names)

if missing_columns:
    raise RuntimeError(f"ImageFolder 열 생성 실패: {missing_columns}")

sample_split = "validation" if "validation" in dataset else "train"
sample = dataset[sample_split][0]

print(sample["id"])
print(sample["text"])
display(sample["conditioning_image"].resize((384, 384)))
display(sample["image"].resize((384, 384)))


In [ ]:

from accelerate.utils import write_basic_config

write_basic_config(mixed_precision=MIXED_PRECISION)
print("Accelerate 설정 완료")


In [ ]:

def run_training_command(command):
    print(shlex.join(command))
    environment = os.environ.copy()
    environment["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    subprocess.run(command, check=True, env=environment)

validation_record = validation_records[0]
VALIDATION_SKETCH = DATASET_DIR / "validation" / validation_record["conditioning_image_file_name"]
VALIDATION_PROMPT = validation_record["text"]

print("VALIDATION_SKETCH:", VALIDATION_SKETCH)
print("VALIDATION_PROMPT:", VALIDATION_PROMPT)



## 1단계: 한국 화풍 LoRA 학습

원본 이미지와 설명만 사용해 SD v1.5 UNet의 LoRA를 학습합니다. 동일한 스타일 문구를 모든 캡션 앞에 붙여 추론 시 해당 문구로 스타일을 호출하도록 구성했습니다.


In [ ]:
!rm -rf /content/diffusers
!git clone --branch v0.39.0 --depth 1 https://github.com/huggingface/diffusers.git /content/diffusers
!pip install -q -e /content/diffusers
!pip install -q -r /content/diffusers/examples/text_to_image/requirements.txt
!pip install -q bitsandbytes tensorboard

from accelerate.utils import write_basic_config
write_basic_config()

In [ ]:
import os
import gc
import sys
import shlex
import subprocess
import importlib.util
import torch
from pathlib import Path
from datetime import datetime

SMOKE_TEST_STEPS = 20
SMOKE_CHECKPOINTING_STEPS = 10

if importlib.util.find_spec("torchao") is not None:
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

if importlib.util.find_spec("tensorboard") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "tensorboard"]
    )

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

DIFFUSERS_REPO = Path("/content/diffusers")
lora_script = DIFFUSERS_REPO / "examples/text_to_image/train_text_to_image_lora.py"
train_data_dir = Path(DATASET_DIR) / "train"
metadata_path = train_data_dir / "metadata.jsonl"

smoke_name = datetime.now().strftime("lora_smoke_%Y%m%d_%H%M%S")
SMOKE_OUTPUT_DIR = Path(LORA_OUTPUT_DIR).parent / smoke_name

if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임이 연결되어 있지 않습니다.")

if not lora_script.exists():
    raise FileNotFoundError(f"학습 스크립트가 없습니다: {lora_script}")

if not train_data_dir.exists():
    raise FileNotFoundError(f"학습 데이터가 없습니다: {train_data_dir}")

if not metadata_path.exists():
    raise FileNotFoundError(f"metadata.jsonl이 없습니다: {metadata_path}")

if bool(globals().get("USE_8BIT_ADAM", True)):
    if importlib.util.find_spec("bitsandbytes") is None:
        raise ImportError("bitsandbytes가 설치되어 있지 않습니다.")

SMOKE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

lora_command = [
    "accelerate",
    "launch",
    "--mixed_precision",
    str(MIXED_PRECISION),
    str(lora_script),
    "--pretrained_model_name_or_path",
    str(BASE_MODEL),
    "--train_data_dir",
    str(train_data_dir),
    "--image_column",
    "image",
    "--caption_column",
    "text",
    "--output_dir",
    str(SMOKE_OUTPUT_DIR),
    "--resolution",
    str(RESOLUTION),
    "--center_crop",
    "--train_batch_size",
    str(TRAIN_BATCH_SIZE),
    "--gradient_accumulation_steps",
    str(GRADIENT_ACCUMULATION_STEPS),
    "--max_train_steps",
    str(SMOKE_TEST_STEPS),
    "--learning_rate",
    str(LORA_LR),
    "--lr_scheduler",
    "constant",
    "--lr_warmup_steps",
    "0",
    "--snr_gamma",
    "5.0",
    "--rank",
    str(LORA_RANK),
    "--checkpointing_steps",
    str(SMOKE_CHECKPOINTING_STEPS),
    "--checkpoints_total_limit",
    "2",
    "--mixed_precision",
    str(MIXED_PRECISION),
    "--gradient_checkpointing",
    "--dataloader_num_workers",
    "0",
    "--report_to",
    "tensorboard",
    "--seed",
    str(SEED),
]

if bool(globals().get("USE_8BIT_ADAM", True)):
    lora_command.append("--use_8bit_adam")

if bool(globals().get("USE_XFORMERS", False)):
    if importlib.util.find_spec("xformers") is not None:
        lora_command.append("--enable_xformers_memory_efficient_attention")
    else:
        print("xformers가 없어 해당 옵션을 제외합니다.")

if torch.cuda.get_device_capability(0)[0] >= 8:
    lora_command.append("--allow_tf32")

def run_training_command(command):
    print("GPU:", torch.cuda.get_device_name(0))
    print("학습 스크립트:", lora_script)
    print("학습 데이터:", train_data_dir)
    print("테스트 step:", SMOKE_TEST_STEPS)
    print("저장 경로:", SMOKE_OUTPUT_DIR)
    print()
    print(shlex.join(command))
    print()

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["TOKENIZERS_PARALLELISM"] = "false"

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    output_lines = []

    if process.stdout is not None:
        for line in process.stdout:
            print(line, end="", flush=True)
            output_lines.append(line)

    return_code = process.wait()

    if return_code != 0:
        recent_log = "".join(output_lines[-120:])
        raise RuntimeError(
            f"학습 프로세스가 종료 코드 {return_code}로 실패했습니다.\n\n"
            f"마지막 로그:\n{recent_log}"
        )

    print()
    print("LoRA 테스트 학습 완료")
    print("저장 경로:", SMOKE_OUTPUT_DIR)

    saved_files = [
        path
        for path in SMOKE_OUTPUT_DIR.rglob("*")
        if path.is_file()
    ]

    if not saved_files:
        raise RuntimeError("학습은 종료됐지만 저장된 파일을 찾지 못했습니다.")

    for path in saved_files:
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"{path} | {size_mb:.2f} MB")

run_training_command(lora_command)


## 2단계: 스케치 ControlNet 학습

원본 이미지가 target, 동일 파일명의 스케치가 conditioning image, txt가 caption으로 사용됩니다. Canny 기반 SD v1.5 ControlNet에서 시작해 현재 스케치 도메인에 맞게 미세조정합니다.


In [ ]:
import os
import gc
import sys
import shlex
import subprocess
import importlib.util
import urllib.request
import torch
from pathlib import Path
from datetime import datetime

SMOKE_TEST_STEPS = 20
SMOKE_CHECKPOINTING_STEPS = 10

if importlib.util.find_spec("tensorboard") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "tensorboard"]
    )

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

DIFFUSERS_REPO = Path("/content/diffusers")
controlnet_script = DIFFUSERS_REPO / "examples/controlnet/train_controlnet.py"
train_data_dir = Path(DATASET_DIR) / "train"
metadata_path = train_data_dir / "metadata.jsonl"

smoke_name = datetime.now().strftime("controlnet_smoke_%Y%m%d_%H%M%S")
CONTROLNET_SMOKE_OUTPUT_DIR = (
    Path(CONTROLNET_OUTPUT_DIR) / "smoke_tests" / smoke_name
)

controlnet_script.parent.mkdir(parents=True, exist_ok=True)

if not controlnet_script.exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/huggingface/diffusers/v0.39.0/examples/controlnet/train_controlnet.py",
        controlnet_script,
    )

if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임이 연결되어 있지 않습니다.")

if not controlnet_script.exists():
    raise FileNotFoundError(
        f"ControlNet 스크립트가 없습니다: {controlnet_script}"
    )

if not train_data_dir.exists():
    raise FileNotFoundError(
        f"학습 데이터 폴더가 없습니다: {train_data_dir}"
    )

if not metadata_path.exists():
    raise FileNotFoundError(
        f"metadata.jsonl이 없습니다: {metadata_path}"
    )

if bool(globals().get("USE_8BIT_ADAM", True)):
    if importlib.util.find_spec("bitsandbytes") is None:
        raise ImportError("bitsandbytes가 설치되어 있지 않습니다.")

CONTROLNET_SMOKE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

controlnet_command = [
    "accelerate",
    "launch",
    "--mixed_precision",
    str(MIXED_PRECISION),
    str(controlnet_script),
    "--pretrained_model_name_or_path",
    str(BASE_MODEL),
    "--controlnet_model_name_or_path",
    str(CONTROLNET_INIT),
    "--train_data_dir",
    str(train_data_dir),
    "--image_column",
    "image",
    "--conditioning_image_column",
    "conditioning_image",
    "--caption_column",
    "text",
    "--output_dir",
    str(CONTROLNET_SMOKE_OUTPUT_DIR),
    "--resolution",
    str(RESOLUTION),
    "--train_batch_size",
    str(TRAIN_BATCH_SIZE),
    "--gradient_accumulation_steps",
    str(GRADIENT_ACCUMULATION_STEPS),
    "--max_train_steps",
    str(SMOKE_TEST_STEPS),
    "--learning_rate",
    str(CONTROLNET_LR),
    "--lr_scheduler",
    "constant",
    "--lr_warmup_steps",
    "0",
    "--checkpointing_steps",
    str(SMOKE_CHECKPOINTING_STEPS),
    "--checkpoints_total_limit",
    "2",
    "--mixed_precision",
    str(MIXED_PRECISION),
    "--gradient_checkpointing",
    "--set_grads_to_none",
    "--dataloader_num_workers",
    "0",
    "--report_to",
    "tensorboard",
    "--seed",
    str(SEED),
]

if bool(globals().get("USE_8BIT_ADAM", True)):
    controlnet_command.append("--use_8bit_adam")

if bool(globals().get("USE_XFORMERS", False)):
    if importlib.util.find_spec("xformers") is not None:
        controlnet_command.append(
            "--enable_xformers_memory_efficient_attention"
        )
    else:
        print("xformers가 없어 해당 옵션을 제외합니다.")

if torch.cuda.get_device_capability(0)[0] >= 8:
    controlnet_command.append("--allow_tf32")

def run_controlnet_training(command):
    print("GPU:", torch.cuda.get_device_name(0))
    print("학습 스크립트:", controlnet_script)
    print("학습 데이터:", train_data_dir)
    print("초기 ControlNet:", CONTROLNET_INIT)
    print("테스트 step:", SMOKE_TEST_STEPS)
    print("저장 경로:", CONTROLNET_SMOKE_OUTPUT_DIR)
    print()
    print(shlex.join(command))
    print()

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["TOKENIZERS_PARALLELISM"] = "false"

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    output_lines = []

    if process.stdout is not None:
        for line in process.stdout:
            print(line, end="", flush=True)
            output_lines.append(line)

    return_code = process.wait()

    if return_code != 0:
        recent_log = "".join(output_lines[-150:])
        raise RuntimeError(
            f"학습 프로세스가 종료 코드 {return_code}로 실패했습니다.\n\n"
            f"마지막 로그:\n{recent_log}"
        )

    print()
    print("ControlNet 테스트 학습 완료")
    print("저장 경로:", CONTROLNET_SMOKE_OUTPUT_DIR)

    saved_files = [
        path
        for path in CONTROLNET_SMOKE_OUTPUT_DIR.rglob("*")
        if path.is_file()
    ]

    if not saved_files:
        raise RuntimeError("학습은 끝났지만 저장 파일이 없습니다.")

    for path in saved_files:
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"{path} | {size_mb:.2f} MB")

run_controlnet_training(controlnet_command)


## 학습 로그 확인

학습 중 `OUTPUT_ROOT` 아래에 TensorBoard 로그와 체크포인트가 저장됩니다.


In [ ]:

print("LoRA 로그:", LORA_OUTPUT_DIR / "logs")
print("ControlNet 로그:", CONTROLNET_OUTPUT_DIR / "logs")



## SD v1.5 + ControlNet + LoRA 결합 추론

검증 스케치 한 장으로 여러 seed 후보를 생성합니다. 이번 단계에서는 evaluator를 적용하지 않고 후보만 저장합니다.


In [ ]:

import gc

from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetPipeline,
    UniPCMultistepScheduler,
)

if not (CONTROLNET_OUTPUT_DIR / "config.json").exists():
    raise FileNotFoundError(f"학습된 ControlNet이 없습니다: {CONTROLNET_OUTPUT_DIR}")

lora_weight_path = LORA_OUTPUT_DIR / "pytorch_lora_weights.safetensors"

dtype = torch.float16

controlnet = ControlNetModel.from_pretrained(
    str(CONTROLNET_OUTPUT_DIR),
    torch_dtype=dtype,
)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    BASE_MODEL,
    controlnet=controlnet,
    torch_dtype=dtype,
    use_safetensors=True,
)

pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

if lora_weight_path.exists():
    pipe.load_lora_weights(str(LORA_OUTPUT_DIR), adapter_name="korean_style")
    pipe.set_adapters(["korean_style"], adapter_weights=[0.9])
else:
    print("LoRA 가중치가 없어 ControlNet만 사용합니다.")

if USE_XFORMERS:
    pipe.enable_xformers_memory_efficient_attention()

pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
pipe.set_progress_bar_config(disable=False)

gc.collect()
torch.cuda.empty_cache()


In [ ]:

CUSTOM_SKETCH_PATH = VALIDATION_SKETCH
CUSTOM_PROMPT = VALIDATION_PROMPT
NEGATIVE_PROMPT = "photorealistic, 3d render, modern western painting, blurry, low quality, watermark, text"
NUM_CANDIDATES = 4
SEEDS = [SEED + index for index in range(NUM_CANDIDATES)]
CONTROLNET_SCALE = 0.9
GUIDANCE_SCALE = 6.5
INFERENCE_STEPS = 30
LORA_SCALE = 0.9

if lora_weight_path.exists():
    pipe.set_adapters(["korean_style"], adapter_weights=[LORA_SCALE])

with Image.open(CUSTOM_SKETCH_PATH) as sketch_image:
    sketch_image = image_to_rgb(sketch_image, (255, 255, 255))
    sketch_image = ImageOps.pad(
        sketch_image,
        (RESOLUTION, RESOLUTION),
        method=Image.Resampling.LANCZOS,
        color=(255, 255, 255),
        centering=(0.5, 0.5),
    )

generated_images = []

for seed in SEEDS:
    generator = torch.Generator(device="cpu").manual_seed(seed)
    image = pipe(
        prompt=CUSTOM_PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        image=sketch_image,
        num_inference_steps=INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        controlnet_conditioning_scale=CONTROLNET_SCALE,
        generator=generator,
        height=RESOLUTION,
        width=RESOLUTION,
    ).images[0]
    output_path = GENERATED_DIR / f"candidate_seed_{seed}.png"
    image.save(output_path)
    generated_images.append(image)

def make_grid(images, columns):
    width, height = images[0].size
    rows = math.ceil(len(images) / columns)
    grid = Image.new("RGB", (columns * width, rows * height), (255, 255, 255))
    for index, image in enumerate(images):
        x = (index % columns) * width
        y = (index // columns) * height
        grid.paste(image, (x, y))
    return grid

grid = make_grid(generated_images, columns=2)
grid_path = GENERATED_DIR / "candidates_grid.png"
grid.save(grid_path)

print("저장 위치:", GENERATED_DIR)
display(sketch_image.resize((384, 384)))
display(grid)


### 학습된 모델 저장 확인 및 백업

학습 스크립트가 종료되면 `OUTPUT_ROOT`에 모델이 저장됩니다. 아래 코드는 최종 결과물을 확인합니다.

In [ ]:
import os

def check_model_files(directory, name):
    print(f"--- {name} 저장 확인 ---")
    if os.path.exists(directory):
        files = os.listdir(directory)
        print(f"경로: {directory}")
        print(f"파일 목록: {files}")
    else:
        print(f"경로가 존재하지 않습니다: {directory}")
    print()

# LoRA 및 ControlNet 저장 결과 확인
check_model_files(LORA_OUTPUT_DIR, "LoRA Weights")
check_model_files(CONTROLNET_OUTPUT_DIR, "ControlNet Model")

# 추가로 특정 버전이나 날짜별로 백업하고 싶을 경우 아래 주석을 해제하여 사용하세요.
# backup_path = f"/content/drive/MyDrive/korean_sketch_model_backup_v1"
# shutil.copytree(OUTPUT_ROOT, backup_path)
# print(f"전체 결과물이 {backup_path}로 백업되었습니다.")

## 저장된 모델 다시 불러오기 (새 세션용)

학습을 이미 마친 후, 나중에 모델만 다시 사용하고 싶을 때 아래 코드를 실행하세요.

In [ ]:
import torch
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline
from google.colab import drive

# 1. 드라이브 마운트 (연결이 끊겼을 경우)
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. 저장된 경로 설정 (이전 설정과 동일해야 함)
LOAD_OUTPUT_ROOT = "/content/drive/MyDrive/korean_sketch_model"
LOAD_LORA_DIR = f"{LOAD_OUTPUT_ROOT}/lora"
LOAD_CONTROLNET_DIR = f"{LOAD_OUTPUT_ROOT}/controlnet"

# 3. 모델 로드
dtype = torch.float16

# 학습된 ControlNet 불러오기
loaded_controlnet = ControlNetModel.from_pretrained(LOAD_CONTROLNET_DIR, torch_dtype=dtype)

# 기본 SD v1.5 불러오기
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    controlnet=loaded_controlnet,
    torch_dtype=dtype
)

# 학습된 LoRA 가중치 적용
lora_path = f"{LOAD_LORA_DIR}/pytorch_lora_weights.safetensors"
if os.path.exists(lora_path):
    pipe.load_lora_weights(LOAD_LORA_DIR, adapter_name="korean_style")
    print("모델 및 LoRA 로드 완료!")

pipe.enable_model_cpu_offload()



## 권장 조정값

- 선을 더 강하게 따르게 하려면 `CONTROLNET_SCALE`을 1.0~1.2로 올립니다.
- 화풍을 더 강하게 적용하려면 `LORA_SCALE`을 1.0 전후로 올립니다.
- 원본 구도 보존이 약하면 ControlNet epoch를 늘리고, 화풍이 약하면 LoRA epoch를 늘립니다.
- 과적합으로 학습 이미지가 그대로 복제되면 epoch와 LoRA scale을 낮춥니다.
- txt가 한국어만으로 되어 있고 의미 반영이 약하면 영어 캡션을 병기하는 편이 안정적입니다.
- 16GB GPU에서 OOM이 발생하면 `ENABLE_TRAIN_VALIDATION=False`, `RESOLUTION=384`, `GRADIENT_ACCUMULATION_STEPS=8` 순으로 조정합니다.


## 공식 참고 자료

- [Hugging Face Diffusers: ControlNet training](https://huggingface.co/docs/diffusers/training/controlnet)
- [Hugging Face Diffusers: ControlNet training example](https://github.com/huggingface/diffusers/blob/v0.39.0/examples/controlnet/README.md)
- [Hugging Face Diffusers: LoRA training](https://huggingface.co/docs/diffusers/training/lora)
- [Hugging Face Datasets: ImageFolder와 다중 이미지 메타데이터](https://huggingface.co/docs/datasets/image_dataset)
- [lllyasviel SD v1.5 Canny ControlNet](https://huggingface.co/lllyasviel/sd-controlnet-canny)